In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from statsmodels.tsa.api import VAR

# ============================================================
# VAR Lag Selection
# ------------------------------------------------------------
# 목적
# - lag 1 ~ MAX_LAGS 범위에서
#   AIC / HQIC / FPE / BIC를 모두 비교
# - 각 기준별 추천 lag 정리
# - 다수결 기반 추천 lag 제시
#
# 입력
# - tvpvar_input_scaled.csv
#
# 출력
# - lag_selection_table.csv
# - lag_selection_summary.txt
# ============================================================

# -----------------------------
# Config
# -----------------------------
BASE_DIR = Path("./")

INPUT_FILE = BASE_DIR / "tvpvar_input_scaled.csv"
OUT_TABLE = BASE_DIR / "lag_selection_table.csv"
OUT_SUMMARY = BASE_DIR / "lag_selection_summary.txt"

MAX_LAGS = 10

VAR_NAMES = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "d_UST10Y",
    "d_VIX"
]

DATE_CANDIDATES = ["Date", "date", "DATE"]


# -----------------------------
# Helpers
# -----------------------------
def find_date_col(df: pd.DataFrame):
    for c in DATE_CANDIDATES:
        if c in df.columns:
            return c
    return None


def mode_smallest(values):
    """
    최빈값 반환.
    동률이면 더 작은 lag 반환.
    """
    s = pd.Series(values)
    counts = s.value_counts()
    max_count = counts.max()
    winners = counts[counts == max_count].index.tolist()
    return int(min(winners)), counts.to_dict()


# -----------------------------
# Load
# -----------------------------
df = pd.read_csv(INPUT_FILE)

date_col = find_date_col(df)
if date_col is not None:
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.sort_values(date_col).reset_index(drop=True)
    df = df.drop(columns=[date_col])

df = df[VAR_NAMES].dropna().reset_index(drop=True)

print("Using columns:", VAR_NAMES)
print("Data shape:", df.shape)

# -----------------------------
# Lag Selection
# -----------------------------
model = VAR(df)
sel = model.select_order(maxlags=MAX_LAGS)

# statsmodels 결과는 lag=0부터 들어있을 수 있음
# 여기서는 lag>=1만 사용
lags = list(range(1, MAX_LAGS + 1))

table = pd.DataFrame({
    "lag": lags,
    "aic": sel.ics["aic"][1:MAX_LAGS + 1],
    "bic": sel.ics["bic"][1:MAX_LAGS + 1],
    "hqic": sel.ics["hqic"][1:MAX_LAGS + 1],
    "fpe": sel.ics["fpe"][1:MAX_LAGS + 1],
})

# 각 정보기준별 최소 lag
best_aic = int(table.loc[table["aic"].idxmin(), "lag"])
best_bic = int(table.loc[table["bic"].idxmin(), "lag"])
best_hqic = int(table.loc[table["hqic"].idxmin(), "lag"])
best_fpe = int(table.loc[table["fpe"].idxmin(), "lag"])

# 다수결 추천 lag
vote_list = [best_aic, best_bic, best_hqic, best_fpe]
recommended_lag, vote_count_dict = mode_smallest(vote_list)

# 표에 추천 여부 표시
table["best_aic"] = table["lag"] == best_aic
table["best_bic"] = table["lag"] == best_bic
table["best_hqic"] = table["lag"] == best_hqic
table["best_fpe"] = table["lag"] == best_fpe
table["recommended_by_vote"] = table["lag"] == recommended_lag

# 보기 좋게 반올림
table_to_save = table.copy()
for col in ["aic", "bic", "hqic", "fpe"]:
    table_to_save[col] = table_to_save[col].astype(float)

table_to_save.to_csv(OUT_TABLE, index=False, encoding="utf-8-sig")

# -----------------------------
# Summary Text
# -----------------------------
summary_lines = []
summary_lines.append("============================================================")
summary_lines.append("Lag Selection Summary")
summary_lines.append("============================================================")
summary_lines.append("")
summary_lines.append(f"Input file : {INPUT_FILE}")
summary_lines.append(f"Variables  : {', '.join(VAR_NAMES)}")
summary_lines.append(f"Sample size: {len(df)}")
summary_lines.append(f"Max lags   : {MAX_LAGS}")
summary_lines.append("Candidate lags: 1 ~ {}".format(MAX_LAGS))
summary_lines.append("")
summary_lines.append("Best lag by criterion")
summary_lines.append("---------------------")
summary_lines.append(f"AIC : {best_aic}")
summary_lines.append(f"BIC : {best_bic}")
summary_lines.append(f"HQIC: {best_hqic}")
summary_lines.append(f"FPE : {best_fpe}")
summary_lines.append("")
summary_lines.append("Vote counts")
summary_lines.append("-----------")
for lag, cnt in sorted(vote_count_dict.items()):
    summary_lines.append(f"lag {lag}: {cnt} vote(s)")
summary_lines.append("")
summary_lines.append(f"Recommended lag (majority vote): {recommended_lag}")
summary_lines.append("")
summary_lines.append("Decision rule")
summary_lines.append("-------------")
summary_lines.append("- Candidate models were restricted to lag >= 1.")
summary_lines.append("- Final recommendation is based on AIC, BIC, HQIC, and FPE jointly.")
summary_lines.append("- If needed, compare residual diagnostics for nearby lag candidates.")
summary_lines.append("")

with open(OUT_SUMMARY, "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))

# -----------------------------
# Print
# -----------------------------
print("\nLag selection table:")
print(table_to_save.to_string(index=False))

print("\nBest lag by criterion:")
print(f"AIC : {best_aic}")
print(f"BIC : {best_bic}")
print(f"HQIC: {best_hqic}")
print(f"FPE : {best_fpe}")

print("\nVote counts:")
for lag, cnt in sorted(vote_count_dict.items()):
    print(f"lag {lag}: {cnt} vote(s)")

print(f"\nRecommended lag (majority vote): {recommended_lag}")

print("\nSaved:")
print(f" - {OUT_TABLE}")
print(f" - {OUT_SUMMARY}")